# Downloading Dataset

In [5]:
!pip install aicrowd-cli

     |████████████████████████████████| 49 kB 5.7 MB/s eta 0:00:011
     |████████████████████████████████| 205 kB 6.6 MB/s eta 0:00:01
     |████████████████████████████████| 75 kB 4.8 MB/s eta 0:00:01
  Using cached toml-0.10.2-py2.py3-none-any.whl (16 kB)
     |████████████████████████████████| 166 kB 5.0 MB/s eta 0:00:01
  Using cached requests-2.25.1-py2.py3-none-any.whl (61 kB)
     |████████████████████████████████| 54 kB 6.8 MB/s  eta 0:00:01
     |████████████████████████████████| 63 kB 7.0 MB/s  eta 0:00:01
  Using cached dataclasses-0.8-py3-none-any.whl (19 kB)
     |████████████████████████████████| 51 kB 9.7 MB/s  eta 0:00:01
  Attempting uninstall: requests
    Found existing installation: requests 2.24.0
    Uninstalling requests-2.24.0:
      Successfully uninstalled requests-2.24.0
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.42.0
    Uninstalling tqdm-4.42.0:
      Successfully uninstalled tqdm-4.42.0
  Attempting uninstall: toml
    Found exis

In [6]:
API_KEY = 'cc0a3da7611cfc6098a7bd9db11b3ecf' # Please get your your API Key from [https://www.aicrowd.com/participants/me]
!aicrowd login --api-key $API_KEY

API Key valid
Saved API Key successfully!


In [7]:
# Downloading the Dataset
!mkdir data
!aicrowd dataset download --challenge emotion-detection -j 3 -o data

test.csv:   0%|                                      | 0.00/642k [00:00<?, ?B/s]
val.csv:   0%|                                       | 0.00/262k [00:00<?, ?B/s]

train.csv:   0%|                                    | 0.00/2.30M [00:00<?, ?B/s]
val.csv: 100%|████████████████████████████████| 262k/262k [00:00<00:00, 392kB/s]
test.csv: 100%|███████████████████████████████| 642k/642k [00:00<00:00, 676kB/s]


train.csv:  46%|████████████▊               | 1.05M/2.30M [00:01<00:01, 868kB/s]

train.csv: 100%|███████████████████████████| 2.30M/2.30M [00:01<00:00, 1.46MB/s]


# Downloading & Importing Libraries

In [1]:
!pip install --upgrade spacy rich
!python -m spacy download en_core_web_sm # Downloaing the model for engligh language will contains many pretrained preprocessing pipelines

     |████████████████████████████████| 12.4 MB 5.3 MB/s eta 0:00:01
     |████████████████████████████████| 2.3 MB 16.3 MB/s eta 0:00:01
     |████████████████████████████████| 5.8 MB 7.9 MB/s eta 0:00:01
     |████████████████████████████████| 591 kB 11.7 MB/s eta 0:00:01
     |████████████████████████████████| 42 kB 4.8 MB/s  eta 0:00:01
     |████████████████████████████████| 449 kB 21.8 MB/s eta 0:00:01
     |████████████████████████████████| 106 kB 19.7 MB/s eta 0:00:01
  Using cached smart_open-3.0.0-py3-none-any.whl
  Using cached contextvars-2.4-py3-none-any.whl
  Using cached immutables-0.15-cp36-cp36m-macosx_10_14_x86_64.whl (53 kB)
  Attempting uninstall: smart-open
    Found existing installation: smart-open 2.0.0
    Uninstalling smart-open-2.0.0:
      Successfully uninstalled smart-open-2.0.0
     |████████████████████████████████| 13.7 MB 9.2 MB/s eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [5]:
!curl https://raw.githubusercontent.com/tylerneylon/explacy/master/explacy.py -o explacy.py

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  6896  100  6896    0     0   164k      0 --:--:-- --:--:-- --:--:--  164k


In [6]:
import time
import pandas as pd
import spacy
import explacy
import random
from sklearn import tree
from sklearn.metrics import f1_score, accuracy_score
import os

# To make things more beautiful! 
from rich.console import Console
from rich.table import Table
from rich import pretty
pretty.install()


# Seeding everything for getting same results 
random.seed(1)
spacy.util.fix_random_seed(1)

In [7]:
# spaCy v3.0 the the latest version spaCy 
spacy.__version__

'3.0.6'

# Reading Dataset

In [8]:
train_dataset = pd.read_csv("data/train.csv")
validation_dataset = pd.read_csv("data/val.csv")[1:]
test_dataset = pd.read_csv("data/test.csv")
train_dataset.head(20)

,text,label
0,takes no time to copy/paste a press release,0
1,You're delusional,1
2,Jazz fan here. I completely feel. Lindsay Mann...,0
3,ah i was also confused but i think they mean f...,0
4,Thank you so much. ♥️ that means a lot.,0
5,And I’ll be there!!!,0
6,There are some amazingly cringey compilations ...,0
7,Check the frame (FPS) limit option in the adva...,0
8,you made me think I was in the dbd subreddit w...,0
9,It was in your op.,0


In [18]:
train_dataset['label'].value_counts()

# Text Classification

In [9]:
nlp = spacy.load('en_core_web_sm')

In [10]:
# Getting a sample text from training dataset to demonstrate word2vec  
sample_text = train_dataset.iloc[2]['text'] 
sample_text

"Jazz fan here. I completely feel. Lindsay Mann cousins has more votes than Lindsay Mann, and Lindsay Mann hasn't even stepped on the court this year"

In [11]:
# Inputting the text in nlp function
doc = nlp(sample_text)

# Getting the embeddings from the sample text
doc.vector

array([ 0.5871044 ,  0.10283045,  0.23638554, -0.08171239,  0.02029083,
       -0.12000011, -0.1948305 ,  0.1250714 , -0.01082261, -0.34358275,
       -0.16639529, -0.04950966, -0.01394221,  0.06337869, -0.30135745,
        0.22211872, -0.17156254,  0.03178046,  0.30427024, -0.10826215,
       -0.25342506,  0.30617625, -0.17000276,  0.35598457, -0.00835321,
       -0.11478721, -0.1430562 ,  0.02518663,  0.60922873,  0.19284171,
       -0.23238468, -0.27463096, -0.13183063, -0.27534658,  0.25664884,
        0.05174499,  0.18620381,  0.11441176, -0.10955156,  0.29338667,
       -0.15877348, -0.02914245,  0.1963947 , -0.04410601,  0.12061837,
       -0.0941467 ,  0.27903876, -0.09223508, -0.00497099, -0.25587952,
        0.21098505,  0.01725493, -0.29827487,  0.0894304 ,  0.14340732,
       -0.0376591 , -0.3396481 ,  0.19914041,  0.28556582,  0.18212257,
        0.5140986 ,  0.02056837, -0.18578346, -0.28987882, -0.16651031,
       -0.10539112, -0.05578137,  0.00634063,  0.02737209,  0.14916842,
        0.15076284,  0.31409967, -0.06142968, -0.13555318, -0.08603293,
        0.40901124, -0.07265005, -0.19719984, -0.349496  ,  0.11685906,
        0.20542377,  0.1133521 ,  0.13061962,  0.2739835 , -0.00384022,
       -0.21771309, -0.28375924, -0.41814512, -0.42588463, -0.06813539,
       -0.27145588,  0.17521115, -0.15065633, -0.05529505,  0.06760314,
       -0.1013436 ], dtype=float32)

# Creating our Dataset

In [12]:
def create_data(dataset, is_train=True):

  # If we are using a training dataset
  if is_train == True:

    # Getting all text into a python list
    texts = list(dataset['text'].values)
                 
    # Put the list into the nlp pipeline and converting the output into a list
    preprocessed_texts = list(nlp.pipe(texts))

    # Getting vectors for all texts 
    X = [string.vector  for string in preprocessed_texts]

    # Labels for the corrosponding texts 
    y = dataset['label'].tolist()

    return X, y

  else:

    # Getting all text into a python list
    texts = list(dataset['text'].values)
                 
    # Put the list into the nlp pipeline and converting the output into a list
    preprocessed_texts = list(nlp.pipe(texts))

    # Getting vectors for all texts 
    X = [string.vector  for string in preprocessed_texts]

    return X

In [15]:
# Creating the training dataset
start_time = time.time()
X_train, y_train = create_data(train_dataset)
print("Elapsed time: %s seconds" % round(time.time() - start_time, 4))

# Creating the validation dataset
start_time = time.time()
X_val, y_val = create_data(validation_dataset)
print("Elapsed time: %s seconds" % round(time.time() - start_time, 4))

X_train[0], y_train[0]

Elapsed time: 47.9502 seconds
Elapsed time: 5.5723 seconds


(
    array([ 0.14151856, -0.05933758, -0.08044346, -0.1107378 ,  0.11325426,
        0.15893325, -0.44572473,  0.21314225,  0.07863858, -0.12423076,
        0.08870672, -0.16083065,  0.03217425, -0.18913928, -0.43932933,
        0.614143  , -0.04348904,  0.15507331, -0.10762676, -0.6841317 ,
       -0.52705824,  0.19307005,  0.5150874 , -0.78364027, -0.5798768 ,
       -0.36855024,  0.261396  ,  0.08007441,  0.13464396, -0.11683945,
       -0.40335947, -0.4810744 , -0.1996509 , -0.40538788,  0.907061  ,
       -0.08425845,  0.41184407, -0.05256712, -0.01928839,  0.67991185,
        0.18288395, -0.00932413,  0.15208176,  0.5218999 , -0.21960959,
       -0.0870916 ,  0.0571377 ,  0.39526838,  0.11505216,  0.03575191,
        0.18940884,  0.35989684, -0.03341857,  0.50211823,  0.25563306,
       -0.45388022,  0.04130013,  0.1111506 ,  0.11391255,  0.12924205,
       -0.44648582, -0.04701398, -0.12538372,  0.11663225,  0.3620197 ,
       -0.00661604, -0.25024778, -0.3998903 , -0.07583453,  0.747198  ,
        0.5959583 , -0.17592818, -0.04306039, -0.52941674, -0.12472894,
       -0.43728942, -0.06499843,  0.0685156 ,  0.23714057, -0.20262413,
        0.41856593,  0.01466806, -0.19088936,  0.36778176,  0.09763627,
       -0.3632294 , -0.15831968, -0.43174067, -0.08282916, -0.09869532,
        0.14616643,  0.02205803, -0.33136448,  0.19892709, -0.50562334,
       -0.14163868], dtype=float32),
    0
)

# Creating the Model

In [19]:
clf = tree.DecisionTreeClassifier()

# Training

In [20]:
start_time = time.time()
clf = clf.fit(X_train, y_train)
print("Elapsed time: %s seconds" % round(time.time() - start_time, 4))

clf

Elapsed time: 4.7165 seconds


DecisionTreeClassifier()

# Validation

In [21]:
y_pred = clf.predict(X_val)

# Getting F1 & Accuracy score of validation predictions
f1 = f1_score(y_val, y_pred)
accuracy = accuracy_score(y_val, y_pred)

print(f"Validation F1 Score  : {round(f1, 4)} and Accuracy Score {round(accuracy, 4)}")

Validation F1 Score  : 0.2493 and Accuracy Score 0.6757


# Submitting Results

In [22]:
# By settings is_train=False, the create_data function will only output the features as setuped in the function

start_time = time.time()
test_data = create_data(test_dataset, is_train=False)

test_predictions = clf.predict(test_data)
print("Elapsed time: %s seconds" % round(time.time() - start_time, 4))

Elapsed time: 15.7579 seconds


In [23]:
# Applying the predictions to the labels column of the sample submission 
test_dataset['label'] = test_predictions
print(test_dataset.shape)

test_dataset.head(20)

(8682, 2)


,text,label
0,I was already over the edge with Cassie Zamora...,0
1,I think you're right. She has oodles of cash a...,1
2,Haha I love this. I used to give mine phone bo...,0
3,Probably out of desperation as they going no a...,1
4,Sorry !! You’re real good at that!!,0
5,I say we get the pitch forks and make him have...,0
6,He looks really different now.,1
7,I swear people just want to be angry,0
8,lol robot car,0
9,Yeah I’ve been watching these videos and they ...,1


In [24]:
!mkdir assets

# Saving the sample submission in assets directory
test_dataset.to_csv(os.path.join("assets", "submission.csv"), index=False)

# Uploading the Results

In [25]:
!aicrowd notebook submit -c emotion-detection -a assets --no-verify

Using notebook: /Users/administrator/Desktop/shenghao-repos/aicrowd-emotion-detection/starter-code.ipynb for submission...
Scrubbing API keys from the notebook...
submission.zip ━━━━━━━━━━━━━━━━━━━ 100.0% • 306.9/305.3 KB • 40.4 kB/s • 0:00:00 • 0:00:010:00:01
                                                  ╭─────────────────────────╮                                                  
                                                  │ Successfully submitted! │                                                  
                                                  ╰─────────────────────────╯                                                  
                                                        Important links                                                        
┌──────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│  This submission │ https://www.aicrowd.com/challenges/ai-blitz-9/problems/emotion-detection/submi